In [0]:
# =============================================================
# 05_BATCH_INFERENCE — Setup Autónomo (sin %run)
# =============================================================
import json
import boto3
import os
import sys

# --- Build _candidates for credential search ---
_candidates = []
try:
    _nb_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
    _repo_dir = "/Workspace" + str(_nb_path).rsplit("/", 2)[0]
    _candidates.append(_repo_dir)
except Exception:
    pass
_vsc_file = globals().get("__vsc_ipynb_file__", "")
if _vsc_file:
    _nb_dir = os.path.dirname(os.path.abspath(_vsc_file))
    _candidates.append(_nb_dir)
    _parent = os.path.dirname(_nb_dir)
    if _parent and _parent != _nb_dir:
        _candidates.append(_parent)
_candidates.append(os.getcwd())
for _candidate in _candidates:
    if _candidate and _candidate not in sys.path:
        sys.path.insert(0, _candidate)

# --- Credenciales ---
try:
    config = {
        "aws_access_key": dbutils.secrets.get(scope="aws", key="access_key"),
        "aws_secret_key": dbutils.secrets.get(scope="aws", key="secret_key"),
    }
    print("✅ Credenciales desde Databricks Secrets.")
except Exception:
    try:
        _aws_file = next(
            (os.path.join(d, "aws_secrets.json") for d in _candidates
             if d and os.path.isfile(os.path.join(d, "aws_secrets.json"))),
            "aws_secrets.json"
        )
        with open(_aws_file, "r") as f:
            config = json.load(f)
        print("✅ Credenciales desde aws_secrets.json.")
    except FileNotFoundError:
        raise SystemExit("❌ Credenciales no disponibles.")

BUCKET = "bronce-scrap-date"
S3_OPTIONS = {
    "fs.s3a.access.key": config["aws_access_key"],
    "fs.s3a.secret.key": config["aws_secret_key"],
    "fs.s3a.endpoint": "s3.amazonaws.com",
}

print(f"✅ Bucket: {BUCKET}")

In [0]:
import pandas as pd
import numpy as np
import pickle

if os.getcwd() not in sys.path:
    sys.path.append(os.getcwd())

from src.ml.scorer import score_dataframe

GOLD_PATH = f"s3a://{BUCKET}/gold/app_inmuebles/"
SCORED_PATH = f"s3a://{BUCKET}/gold/app_inmuebles_scored/"

print(f"📖 Cargando Parquet desde: {GOLD_PATH}")
df_gold_spark = spark.read.format("parquet").options(**S3_OPTIONS).load(GOLD_PATH)

print("⬇️ Bajando a Pandas...")
df_pandas = df_gold_spark.toPandas()
print(f"✅ {len(df_pandas)} registros cargados.")


In [0]:
%pip install xgboost

In [0]:
import io

print(f"🔍 Buscando modelo en s3://{BUCKET}/models/ ...")

s3_client = boto3.client(
    's3',
    aws_access_key_id=config["aws_access_key"],
    aws_secret_access_key=config["aws_secret_key"],
    region_name='us-east-1'
)

response = s3_client.list_objects_v2(Bucket=BUCKET, Prefix='models/')
if 'Contents' not in response:
    raise FileNotFoundError(f"No hay archivos en s3://{BUCKET}/models/")

model_files = [
    obj for obj in response['Contents']
    if obj['Key'].endswith('.pkl') and 'bundle_v' in obj['Key']
]
if not model_files:
    raise FileNotFoundError("No se encontró bundle .pkl en S3.")

latest = max(model_files, key=lambda x: x['LastModified'])
print(f"📦 Descargando: {latest['Key']}")

body = s3_client.get_object(Bucket=BUCKET, Key=latest['Key'])['Body'].read()
bundle = pickle.load(io.BytesIO(body))

print(f"✅ Modelo {bundle.get('model_version', 'v?')} cargado.")


In [0]:
print("⚙️ Generando rentabilidad...")
df_scored_pandas = score_dataframe(df_pandas, bundle)
print("✅ Inferencia completada.")
display(df_scored_pandas[["titulo", "city_token", "precio_num", "precio_predicho", "rentabilidad_potencial", "estado_inversion"]].head(10))


In [0]:
# =============================================================
# ENRICHMENT: Columnas analíticas para la app (post-inference)
# =============================================================
import numpy as np

# 1. Descuento potencial absoluto COP
df_scored_pandas["descuento_potencial_cop"] = (
    df_scored_pandas["precio_predicho"] - df_scored_pandas["precio_num"]
).round(0)

# 2. Precio m2 mediano de ciudad (desde bundle city_stats)
city_stats = bundle.get("city_stats")
if city_stats is not None and not city_stats.empty and "city_token" in city_stats.columns:
    _pm2_city = city_stats[["city_token", "precio_m2_mediano_ciudad"]].drop_duplicates("city_token")
    df_scored_pandas = df_scored_pandas.merge(_pm2_city, on="city_token", how="left")
else:
    df_scored_pandas["precio_m2_mediano_ciudad"] = np.nan

# 3. Precio m2 vs mediana ciudad (%)
if "precio_m2" in df_scored_pandas.columns and "precio_m2_mediano_ciudad" in df_scored_pandas.columns:
    df_scored_pandas["precio_m2_vs_mediana_pct"] = (
        (df_scored_pandas["precio_m2"] / df_scored_pandas["precio_m2_mediano_ciudad"].replace(0, np.nan) - 1) * 100
    ).replace([float("inf"), float("-inf")], np.nan).round(1)

# 4. Percentil de precio m2 dentro de la ciudad (0-100)
if "city_token" in df_scored_pandas.columns and "precio_m2" in df_scored_pandas.columns:
    df_scored_pandas["percentil_precio_ciudad"] = (
        df_scored_pandas.groupby("city_token")["precio_m2"].rank(pct=True) * 100
    ).round(1)

# 5. Score inversión 0-100: rentabilidad (50%) + completitud (30%) + consistencia (20%)
_has_inputs = all(c in df_scored_pandas.columns
                  for c in ["rentabilidad_potencial", "data_completeness", "dispersion_pct_grupo"])
if _has_inputs:
    _rent = (df_scored_pandas["rentabilidad_potencial"].fillna(0).clip(-50, 50) + 50) / 100
    _comp = (df_scored_pandas["data_completeness"].fillna(2.5) / 5.0).clip(0, 1)
    _cons = 1 - df_scored_pandas["dispersion_pct_grupo"].fillna(0).clip(0, 100) / 100
    df_scored_pandas["score_inversion"] = (_rent * 0.50 + _comp * 0.30 + _cons * 0.20).mul(100).round(1)

# 6. Cuota mensual estimada (30 años, tasa 14% EA aprox 1.098%/mes, financiación 70%)
_tm, _n = 0.01098, 360
_fc = (_tm * (1 + _tm)**_n) / ((1 + _tm)**_n - 1)
df_scored_pandas["cuota_mensual_est"] = (df_scored_pandas["precio_num"] * 0.70 * _fc).round(-3)

new_cols = ["descuento_potencial_cop", "precio_m2_mediano_ciudad", "precio_m2_vs_mediana_pct",
            "percentil_precio_ciudad", "score_inversion", "cuota_mensual_est"]
for c in new_cols:
    n_ok = int(df_scored_pandas[c].notna().sum()) if c in df_scored_pandas.columns else 0
    print(f"  {c}: {n_ok} valores validos")
print("Enrichment completado.")


In [0]:
print("⬆️ Guardando resultado...")

# Spark no acepta object-dtype con mezcla de tipos, pero convertir TODOS los
# nulls a "" destruye la semantica de "sin dato": las 50.957 filas sin barrio
# oficial quedaban con barrio="" y en la app `barrio.fillna(barrio_texto)` ya
# no sustituia nada, asi que de 9.176 barrios distintos solo se veian 355.
# Se castea a texto conservando el null.
for col in df_scored_pandas.columns:
    if df_scored_pandas[col].dtype == "object":
        df_scored_pandas[col] = (
            df_scored_pandas[col]
            .astype("object")
            .map(lambda v: None if v is None or (isinstance(v, float) and v != v) else str(v))
        )

df_scored_spark = spark.createDataFrame(df_scored_pandas)

print(f"💾 Guardando en: {SCORED_PATH}")
writer = df_scored_spark.coalesce(1).write.format("parquet").mode("overwrite").options(**S3_OPTIONS)
writer.save(SCORED_PATH)

print("🎉 Misión Inferencia Batch Finalizada.")


In [0]:
# ==============================================================================
# CELDA FINAL: COMPACTACION DE TABLAS GOLD A 1 SOLO ARCHIVO (EVITA FRAGMENTACION)
# ==============================================================================
import os as _os

def compactar_tabla_s3(path_s3, format_type="parquet", options_s3=None):
    if options_s3 is None:
        options_s3 = {}
    try:
        from pyspark.sql import SparkSession
        spark = SparkSession.builder.getOrCreate()
        print(f"Compactando {path_s3} via Spark coalesce(1)...")
        df_sp = spark.read.format(format_type).options(**options_s3).load(path_s3)
        df_sp.coalesce(1).write.format(format_type).mode("overwrite").options(**options_s3).option("overwriteSchema","true").save(path_s3)
        print(f"[Spark] Compactacion completada: {path_s3}")
        return True
    except Exception as e:
        print(f"[Spark] No disponible ({e}). Usando fallback Pandas/PyArrow...")
    try:
        import pandas as pd
        import pyarrow as pa, pyarrow.dataset as ds, pyarrow.parquet as pq, s3fs
        key = options_s3.get("fs.s3a.access.key") or _os.environ.get("AWS_ACCESS_KEY_ID")
        secret = options_s3.get("fs.s3a.secret.key") or _os.environ.get("AWS_SECRET_ACCESS_KEY")
        fs = s3fs.S3FileSystem(key=key, secret=secret)
        clean = path_s3.replace("s3a://","").replace("s3://","")
        print(f"[Fallback] Leyendo desde S3: {clean}")
        tbl = ds.dataset(clean, filesystem=fs, format=format_type).to_table()
        try: fs.rm(clean, recursive=True)
        except: pass
        pq.write_to_dataset(tbl, root_path=clean, filesystem=fs, use_dictionary=True, compression="snappy")
        print(f"[Fallback] Compactacion completada: {path_s3}")
        return True
    except Exception as ex:
        print(f"[Fallback] Fallo: {ex}")
        return False

# --- Ejecucion de Compactacion ---
compactar_tabla_s3(
    path_s3=f"s3a://{BUCKET}/gold/app_inmuebles_scored/",
    format_type="parquet",
    options_s3=S3_OPTIONS
)
